# 05b - Generation experiment: SEA-LION as the generation model

Follow-up on `05_generation.ipynb`. Retrieval is unchanged (BGE-M3 dense/hybrid
search + SEA-LION reranker, same as `05_generation`'s current default). The
only variable that changes is the generation model: from `qwen3:8b` to
`aisingapore/Llama-SEA-LION-v3-8B-IT`.

Why this model: same size class (8B) as qwen3:8b, so the comparison isolates
language specialization rather than parameter count; instruction-tuned (needed
to follow the system prompt); built on Llama 3.1 8B, continued-pretrained and
instruction-tuned specifically on Southeast Asian languages (including Tamil,
Thai, Vietnamese, Burmese) rather than general multilingual coverage. Verified
available to pull directly via Ollama (`ollama pull
aisingapore/Llama-SEA-LION-v3-8B-IT`) before building this notebook.

Note: SEA-LION is doing two different jobs across this repo now -- the
embedding variant (`SEA-LION-E5-Embedding-600M`) reranks retrieval results in
`05_generation`/`06_eval`; this notebook uses a completely different SEA-LION
checkpoint (`Llama-SEA-LION-v3-8B-IT`, a chat/instruct LLM) to write the answer
itself. Same family name, unrelated models -- one is embedding-only, the other
generation-only.

Saves raw `{query, answer, sources}` results to disk so they can be compared
against qwen3:8b's answers later without needing Ollama running again.

## Step 1: Setup

Same as `05_generation.ipynb` -- chunks, BGE-M3 embeddings, Chroma, SEA-LION
reranker -- except `OLLAMA_MODEL_NAME` now points at the SEA-LION generation
model instead of `qwen3:8b`.

In [1]:
import json
from pathlib import Path

import chromadb
import numpy as np

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")
CHROMA_DIR = Path("../data/processed/chroma")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
# Reranker unchanged from 05_generation.ipynb -- retrieval is not a variable
# in this experiment, only the generation model below is.
RERANKER_MODEL_NAME = "aisingapore/SEA-LION-E5-Embedding-600M"
# Experiment: SEA-LION's instruction-tuned generation model, in place of
# qwen3:8b. Same 8B size class, but continued-pretrained and instruction-tuned
# specifically on SEA languages instead of general multilingual data -- see
# notebook intro for the full rationale.
OLLAMA_MODEL_NAME = "aisingapore/Llama-SEA-LION-v3-8B-IT"
OLLAMA_BASE_URL = "http://localhost:11434"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks_by_id = {chunk["chunk_id"]: chunk for chunk in chunks}

embeddings = np.load(EMBEDDINGS_PATH)
chunk_ids = json.loads(CHUNK_IDS_PATH.read_text(encoding="utf-8"))
chunk_id_to_idx = {cid: i for i, cid in enumerate(chunk_ids)}

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="bge_m3")

len(chunks), embeddings.shape, collection.count()

(16, (16, 1024), 16)

## Step 2: Retrieve context

Identical to `05_generation.ipynb` -- same swappable `retrieve_context(query,
method="hybrid_rerank")` using the SEA-LION reranker, unchanged.

In [2]:
import re

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = SentenceTransformer(RERANKER_MODEL_NAME)


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(chunks_by_id[cid]["text"]) for cid in chunk_ids])


def _dense_retrieve(query: str, top_k: int) -> list[str]:
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    results = collection.query(query_embeddings=[query_embedding.tolist()], n_results=top_k)
    return list(results["ids"][0])


def _bm25_retrieve(query: str, top_k: int) -> list[str]:
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [cid for cid, _ in ranked]


def _hybrid_retrieve(query: str, top_k: int) -> list[str]:
    dense_ids = _dense_retrieve(query, top_k=10)
    bm25_ids = _bm25_retrieve(query, top_k=10)
    scores: dict[str, float] = {}
    for ranked in (dense_ids, bm25_ids):
        for rank, cid in enumerate(ranked, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (60 + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]


def _hybrid_rerank_retrieve(query: str, top_k: int) -> list[str]:
    candidates = _hybrid_retrieve(query, top_k=10)
    # STS is the only named prompt documented on the SEA-LION-E5 model card,
    # used for both query and passage since this only needs a symmetric
    # cosine-similarity score for reranking, not asymmetric retrieval.
    query_embedding = reranker_model.encode(query, convert_to_numpy=True, prompt_name="STS")
    candidate_embeddings = reranker_model.encode(
        [chunks_by_id[cid]["text"] for cid in candidates],
        convert_to_numpy=True,
        prompt_name="STS",
    )
    similarities = candidate_embeddings @ query_embedding / (
        np.linalg.norm(candidate_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    reranked = sorted(zip(candidates, similarities), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in reranked[:top_k]]


RETRIEVAL_METHODS = {
    "dense": _dense_retrieve,
    "hybrid": _hybrid_retrieve,
    "hybrid_rerank": _hybrid_rerank_retrieve,
}


def retrieve_context(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> list[dict]:
    retrieved_ids = RETRIEVAL_METHODS[method](query, top_k)
    return [chunks_by_id[cid] for cid in retrieved_ids]


retrieve_context("How much overtime pay am I entitled to?")

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 16748.09it/s]


[{'chunk_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-2',
  'document_id': 'https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days',
  'url': 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days',
  'heading_path': 'Overtime pay',
  'text': 'Hours of work, overtime and rest day > Overtime pay\n\nOvertime work is all work in excess of the normal hours of work (excluding breaks).\n\nYou can claim overtime if you are:\n\n- A non-workman earning a monthly basic salary of $2,600 or less.\n- A workman earning a monthly basic salary of $4,500 or less. \n\nThe overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourly rate of $13.60**.\n\nFor overtime work, your employer must pay you **at least 1.5 times** the hourly basic rate of pay. Payment must be made **within 14 days** after the last day of the salary period.\n\nA non-workman earns $2,600 a month and work

## Step 3: Prompt construction

Identical system prompt and prompt builder to `05_generation.ipynb` -- prompt
wording is not a variable in this experiment, only the model behind it.

In [3]:
SYSTEM_PROMPT = """You are migrantBuddy, an assistant that answers questions about \
Singapore employment rules (work passes, salary, working hours) for migrant workers.

Answer ONLY using the information in the provided context. If the context does not \
contain enough information to answer the question, say so clearly instead of \
guessing. Do not use any outside knowledge. Keep answers clear and concise, \
suitable for someone who may not be a native English speaker."""


def build_prompt(query: str, context_chunks: list[dict]) -> str:
    context_text = "\n\n---\n\n".join(
        f"Source: {chunk['url']}\n{chunk['text']}" for chunk in context_chunks
    )
    return f"""Context:
{context_text}

Question: {query}

Answer:"""


context_chunks = retrieve_context("How much overtime pay am I entitled to?")
prompt = build_prompt("How much overtime pay am I entitled to?", context_chunks)
print(prompt[:1000])

Context:
Source: https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days
Hours of work, overtime and rest day > Overtime pay

Overtime work is all work in excess of the normal hours of work (excluding breaks).

You can claim overtime if you are:

- A non-workman earning a monthly basic salary of $2,600 or less.
- A workman earning a monthly basic salary of $4,500 or less. 

The overtime rate payable for non-workmen is **capped at the salary level of $2,600, or an hourly rate of $13.60**.

For overtime work, your employer must pay you **at least 1.5 times** the hourly basic rate of pay. Payment must be made **within 14 days** after the last day of the salary period.

A non-workman earns $2,600 a month and works 2 hours of overtime. The overtime pay is:

$13.60 × 1.5 × 2 hours = $40.80

Calculate your overtime pay

Overtime pay is calculated as follows:

- Hourly basic rate of pay × 1.5 × number of hours worked overtime

The hourly basic rate of pay is calculated

## Step 4: Ollama call

Same call shape as `05_generation.ipynb`, hitting `OLLAMA_MODEL_NAME` (now
SEA-LION's generation model) instead of qwen3:8b.

In [4]:
import requests


def generate(query: str, method: str = "hybrid_rerank", top_k: int = 5) -> dict:
    context_chunks = retrieve_context(query, method=method, top_k=top_k)
    user_prompt = build_prompt(query, context_chunks)

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL_NAME,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"]

    return {
        "query": query,
        "answer": answer,
        "sources": [chunk["url"] for chunk in context_chunks],
    }


result = generate("How much overtime pay am I entitled to?")
print(result["answer"])

To calculate your overtime pay, follow these steps:

1. Identify if you are:
   - A non-workman earning $2,600/month or less
   - A workman earning $4,500/month or less

2. Calculate the hourly basic rate of pay based on your employment type (monthly-rated, daily-rated, or piece-rated).

3. For overtime worked beyond normal hours, multiply:
   - Hourly basic rate of pay × 1.5 × Number of overtime hours

The maximum overtime allowed in a month is 72 hours.

For example, if you earn $2,600/month and work 2 hours of overtime:
- Hourly rate: $13.60
- Overtime pay: ($13.60 × 1.5 × 2) = $40.80

Payment must be made within 14 days after the salary period.

If your employer requests you to work on a rest day, and it's more than half your normal daily hours, you get double pay plus overtime pay. If you request it, you get:
- For up to half normal daily hours: Half-day’s salary
- More than half normal daily hours: 1 day’s salary + overtime pay

To avoid overpayment or underpayment, verify all ca

## Step 5: End-to-end smoke test

Same 10 sample queries as `05_generation.ipynb`, so answers are directly
comparable question-for-question against the qwen3:8b run.

In [5]:
SAMPLE_QUERIES = [
    {"language": "en", "text": "How much overtime pay am I entitled to?"},
    {"language": "en", "text": "When must my employer pay my salary?"},
    {"language": "ms", "text": "Bilakah majikan saya perlu bayar gaji saya?"},
    {"language": "ta", "text": "எனக்கு எவ்வளவு கூடுதல் நேர ஊதியம் கிடைக்கும்?"},
    {"language": "my", "text": "ကျွန်တော် ဘယ်လောက် အချိန်ပိုခ ရထိုက်သလဲ"},
    {"language": "th", "text": "ฉันมีสิทธิ์ได้รับค่าล่วงเวลาเท่าไหร่"},
    {"language": "vi", "text": "Chủ sử dụng lao động của tôi phải trả lương khi nào?"},
    {"language": "en", "text": "Who pays repatriation costs when my Work Permit ends?"},
    {"language": "en", "text": "How much medical insurance must my employer provide?"},
    {"language": "en", "text": "How can I contact MOM?"},
]

smoke_test_results = []
for query in SAMPLE_QUERIES:
    result = generate(query["text"])
    result["language"] = query["language"]
    smoke_test_results.append(result)

    preview = result["answer"][:300].replace("\n", " ")
    print(f"\n{'=' * 80}\nQuery [{query['language']}]: {query['text']}\n{'=' * 80}")
    print(f"\nAnswer (first 300 chars):\n{preview}...")
    print(f"\nSources: {result['sources']}")


Query [en]: How much overtime pay am I entitled to?

Answer (first 300 chars):
Based on the provided context, you can claim overtime if: 1. You are earning $2,600 or less monthly (non-workman) OR 2. You are a workman earning $4,500 or less monthly  The overtime rate is calculated as follows: - Hourly basic rate × 1.5 × Number of hours worked overtime - Maximum 72 overtime hour...

Sources: ['https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/hours-of-work-overtime-and-rest-days', 'https://www.mom.gov.sg/employment-practices/salary/paying-salary']

Query [en]: When must my employer pay my salary?

Answer (first 300 chars):
According to MOM's guidelines, your employer **must** pay your salary **at least once a month**. However, they can choose sh

## Step 6: Save results

Save to disk so this run can be compared against `05_generation.ipynb`'s
qwen3:8b answers later without needing Ollama running again. Filename includes
the model name so different generation models don't overwrite each other.

In [6]:
import datetime
import re as _re

RESULTS_DIR = Path("../data/processed/eval_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output = {
    "notebook": "05b_generation_sealion",
    "generation_model": OLLAMA_MODEL_NAME,
    "retrieval_method": "hybrid_rerank",
    "generated_at": datetime.datetime.now().isoformat(),
    "results": smoke_test_results,
}

model_slug = _re.sub(r"[^a-zA-Z0-9]+", "_", OLLAMA_MODEL_NAME).strip("_").lower()
output_path = RESULTS_DIR / f"05b_generation_{model_slug}.json"
output_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding="utf-8")
output_path

WindowsPath('../data/processed/eval_results/05b_generation_aisingapore_llama_sea_lion_v3_8b_it.json')